In [1]:
import sys
sys.path.append('..')

import numpy as np
from PIL import Image
from src.elements import *
from src.systems import *
from src.utilities import *

In [2]:
ThinL1 = ThinLens(100)
FreeS = FreeSpace(400)
ThinL2 = ThinLens(500)
System = OpticalSystem()
System.add_element(ThinL1)
System.add_element(FreeS)
System.add_element(ThinL2)
print("System matrix M:")
print(System.M)
print("Focal length of the system:")
print(System.focal_length())
print("Magnification of the system:")
print(System.magnification())

System matrix M:
[[-3.e+00  4.e+02]
 [-4.e-03  2.e-01]]
Focal length of the system:
250.0
Magnification of the system:
-3.0


In [3]:
ThickL1 = ThickLens(51.68, -51.68, 6, 1.5)
FreeSpace = FreeSpace(200)
ThickL2 = ThickLens(155.04, -155.04, 4, 1.5)
System2 = OpticalSystem(color=True)  # Create a new optical system that can handle color
System2.add_element(ThickL1)
System2.add_element(FreeSpace)
System2.add_element(ThickL2)
print("System matrix M for thick lenses:")
print(System2.MRGB)
print("Focal length of the system with thick lenses:")
print(System2.focal_length())
print("Magnification of the system with thick lenses:")
print(System2.magnification())

System matrix M for thick lenses:
[array([[-2.96752455e+00,  1.96894991e+02],
       [ 9.15011212e-05, -3.43052297e-01]]), array([[-2.98610718e+00,  1.96853829e+02],
       [ 2.17362999e-04, -3.49213432e-01]]), array([[-3.02795926e+00,  1.96761598e+02],
       [ 5.05237418e-04, -3.63086563e-01]])]
Focal length of the system with thick lenses:
(np.float64(-10928.827836282668), np.float64(-4600.5990086098745), np.float64(-1979.2674956697679))
Magnification of the system with thick lenses:
(np.float64(-2.9675245453199364), np.float64(-2.9861071819444893), np.float64(-3.0279592642183357))


In [4]:
img_path = "../assets/PIA01464.jpg" 
img = Image.open(img_path)
image_array = np.array(img)
print("Original image shape:", image_array.shape)
pixel_size = 0.01  # Example pixel size in mm
output_image_array = System2.image(image_array, pixel_size)  # Process the image through the optical system
output_image = Image.fromarray(output_image_array)
print("Output image shape:", output_image.size)
output_image.save("../results/saturn_output.jpg")

Original image shape: (256, 371, 3)
Output image shape: (1125, 668)


In [5]:
# Interpolate the image through the system
interpolated_image_array = System2.image_with_interpolation(image_array, pixel_size)
interpolated_image = Image.fromarray(interpolated_image_array)
print("Interpolated image shape:", interpolated_image.size)
interpolated_image.save("../results/saturn_interpolated_output2.jpg")

Interpolated image shape: (1125, 668)


In [6]:
Lens1 = ThickLens(150, -150, 8, 1.5)
Lens2 = ThickLens(-52, 52, 5, 1.5)
Gal_Thick = GalileanTelescope(lens1=Lens1, lens2=Lens2, color=True)  
Gal_Thin_image_array = Gal_Thick.image_infinity(image_array)
Gal_Thick_image_array = Gal_Thick.image_with_interpolation(image_array, infinity=True)
Gal_Thick_image = Image.fromarray(Gal_Thick_image_array)
print("Galilean telescope with thick lenses output image shape:", Gal_Thick_image.size)
Gal_Thick_image.save("../results/saturn_galilean_thick_output.jpg")
print("Galilean telescope with thick lenses system matrix M:")
print(Gal_Thick.MRGB)

Galilean telescope with thick lenses output image shape: (799, 642)
Galilean telescope with thick lenses system matrix M:
[array([[ 2.88609718e-01,  1.10260402e+02],
       [-9.61718619e-04,  3.09747199e+00]]), array([[ 2.85145968e-01,  1.10250805e+02],
       [-1.03352982e-03,  3.10736467e+00]]), array([[ 2.77341782e-01,  1.10229289e+02],
       [-1.19766935e-03,  3.12964658e+00]])]


In [7]:
R1_o = 153.7
R2_o = -153.7
R3_o = -706.8
d1_o = 8.4
d2_o = 7
f_o = 300

R1_e = -42.0
R2_e = 39.5
R3_e = 207.2
d1_e = 2
d2_e = 4.5
f_e = -50

n_reference = refractive_index_NBK7(0.5876)
n_ref_NSF2 = refractive_index_NSF2(0.5876)

Doublet1 = Doublet(R1_o, R2_o, R3_o, d1_o, d2_o, n_reference, n_ref_NSF2)
Doublet2 = Doublet(R1_e, R2_e, R3_e, d1_e, d2_e, n_reference, n_ref_NSF2)

Doublet_telescope = GalileanTelescope(lens1=Doublet1, lens2=Doublet2, color=True, material1=['NBK7', 'NSF2'], material2=['NBAF10', 'NSF6HT'])
Doublet_image_array = Doublet_telescope.image_infinity(image_array)
Doublet_image = Image.fromarray(Doublet_image_array)
print("Galilean telescope with doublet lenses output image shape:", Doublet_image.size)
Doublet_image.save("../results/saturn_doublet_output.jpg")
print("Galilean telescope with doublet lenses system matrix M:")
print(Doublet_telescope.MRGB)

Galilean telescope with doublet lenses output image shape: (1185, 952)
Galilean telescope with doublet lenses system matrix M:
[array([[2.51779005e-01, 2.34348945e+02],
       [7.12813121e-04, 4.63520382e+00]]), array([[2.50896322e-01, 2.34287677e+02],
       [7.00358058e-04, 4.63970636e+00]]), array([[2.50025574e-01, 2.34125668e+02],
       [6.88473424e-04, 4.64428211e+00]])]


In [8]:
# My telescope

image_path = "../assets/jupiter-through-a-16-telescope-v0-8w9mx1hyqxwa1.webp"
image_array = np.array(Image.open(image_path))
obj_lens = ThickLens(206.72, -np.inf, 8.5, refractive_index(0.5876, 'NBK7'))
eye_lens = ThickLens(5.17, -np.inf, 3.7, refractive_index(0.5876, 'NBK7'))
My_telescope = GalileanTelescope(lens1=obj_lens, lens2=eye_lens, color=True)
My_image_array = My_telescope.image_with_interpolation(image_array, infinity=True)
My_image = Image.fromarray(My_image_array)
print("My Galilean telescope output image shape:", My_image.size)
My_image.save("../results/my_galilean_telescope_output.jpg")
print("My Galilean telescope system matrix M:")
print(My_telescope.MRGB)


My Galilean telescope output image shape: (24472, 22053)
My Galilean telescope system matrix M:
[array([[-3.18619153e-02,  3.17038047e+02],
       [ 9.00574910e-04, -4.03464920e+01]]), array([[-3.56032755e-02,  3.16706368e+02],
       [ 1.40042675e-03, -4.05446985e+01]]), array([[-4.40038350e-02,  3.15963149e+02],
       [ 2.54387385e-03, -4.09912089e+01]])]
